# How popular were the top most recent Kdramas around the world in the month of September 2026?

## Task 3

**Table of Content**
1. [Collect a list of the new, popular K-dramas](#sec1)
2. [Find the wikipedia page titles for each show](#sec2)
3. [Get the wikipedia pageviews data](#sec3)

<h3>Import requirements</h3>

In [32]:
import requests
from bs4 import BeautifulSoup
from seleniumbase import Driver
from selenium.webdriver.common.by import By
import re
from mwclient import Site
import time
from mwviews.api import PageviewsClient
import pandas as pd

<h2 id="sec1">1. Collect a list of the new, popular K-dramas</h2>

In [2]:
url = "https://mydramalist.com/search?adv=titles&ty=68&co=3&re=2026,2026&ep=1,16&rt=8.5,10&so=newest&or=desc"

with Driver(browser="firefox") as driver:
    driver.open(url)
    rendered_html = driver.get_page_source()
    soup = BeautifulSoup(rendered_html, "html.parser")
    tvshow_cards = soup.find_all(class_="text-primary title")

print(len(tvshow_cards))

10


In [3]:
print(tvshow_cards[0])

<h6 class="text-primary title"><a href="/795728-four-hands">Four Hands, Two Sonatas</a>
<a class="btn simple btn-manage-list" data-id="795728" data-stats="mylist:795728" rel="nofollow"><span><i class="far fa-plus"></i></span></a>
</h6>


In [20]:
titles = []
for tvshow in tvshow_cards[:10]:
    title_tag = tvshow.find_all("a")
    titles.append(title_tag[0].text)

for i in range(0,10):
    print(f"{i}: {titles[i]}")

0: Four Hands, Two Sonatas
1: A Shop for Killers Season 2
2: Agent Kim Reactivated
3: The Prosecutor's Proposal
4: Teach You a Lesson
5: The WONDERfools
6: The Legend of Kitchen Soldier
7: We Are All Trying Here
8: Yumi's Cells Season 3
9: Bloodhounds Season 2


<h3>Remove season numbers and caps lock to get final show titles</h3>

In [21]:
titles = [str(re.sub(r'season|\d', '', title, flags=re.IGNORECASE)).rstrip() for title in titles]

<h3>The capital letters in WONDERfools will not work with mwclient searching, so remove them</h3>

In [22]:
titles[5] = "The Wonderfools"

In [23]:
for i in range(0,10):
    print(f"{i}: {titles[i]}")

0: Four Hands, Two Sonatas
1: A Shop for Killers
2: Agent Kim Reactivated
3: The Prosecutor's Proposal
4: Teach You a Lesson
5: The Wonderfools
6: The Legend of Kitchen Soldier
7: We Are All Trying Here
8: Yumi's Cells
9: Bloodhounds


<h2 id="sec2">2. Find the wikipedia page titles for each show</h2>

In [9]:
site = Site("en.wikipedia.org")

<h3>Use input to confirm whether the titles lead to the correct wikipedia page with mwclient, not sure how else to confirm than to check</h3>

In [24]:
wikipedia_titles = []

i = 0
for title in titles:
    print(f"{i}: {title}")
    page = site.pages[title]
    text = page.text()
    print(text[:100])
    if "South Korean television series" in text:
        print(">> Title is correct")
        wikipedia_titles.append(title)
    else:
        print("Title is incorrect... trying again")
        title = f"{title} (TV series)"
        time.sleep(2)
        print()
        print(f"{i}: {title} | RETRY")
        page = site.pages[title]
        text = page.text()
        print(text[:100])
        if "South Korean television series" in text:
            print(">> Title + (TV series) is correct")
            wikipedia_titles.append(title)
    print()
    i += 1
    time.sleep(2)

0: Four Hands, Two Sonatas
{{Short description|2026 South Korean television series}}
{{Other uses|Four Hands (disambiguation){{
>> Title is correct

1: A Shop for Killers
{{Short description|2024 South Korean television series}}
{{Distinguish|The Killer's Shopping List}}
>> Title is correct

2: Agent Kim Reactivated
{{Short description|2026 South Korean television series}}
{{Infobox television
| image              
>> Title is correct

3: The Prosecutor's Proposal
{{Short description|2026 South Korean television series}}
{{Infobox television
| image              
>> Title is correct

4: Teach You a Lesson
{{Short description|2026 South Korean television series}}
{{Use mdy dates|date=June 2026}}
{{Infobox
>> Title is correct

5: The Wonderfools
{{Short description|2026 South Korean television series}}
{{Use mdy dates|date=May 2026}}
{{Infobox 
>> Title is correct

6: The Legend of Kitchen Soldier
{{Short description|2026 South Korean television series}}
{{Use mdy dates|date=April 2026}}

In [26]:
for i in range(0,10):
    print(f"{i}: {wikipedia_titles[i]}")

0: Four Hands, Two Sonatas
1: A Shop for Killers
2: Agent Kim Reactivated
3: The Prosecutor's Proposal
4: Teach You a Lesson
5: The Wonderfools
6: The Legend of Kitchen Soldier
7: We Are All Trying Here
8: Yumi's Cells
9: Bloodhounds (TV series)


<h2 id="sec3">3. Get the wikipedia pageviews data</h2>

<h3>First, we need to get all the different language-specific wikipedias</h3>

In [44]:
page = site.pages["List of Wikipedias"]
text = page.text()

In [ ]:
text

'{{Short description|none}}\n{{Use dmy dates|date=September 2018}}\n[[File:Most popular edition of Wikipedia by country.svg|thumb|upright=2|alt=Map of the most popular edition of Wikipedia by country|Most popular edition of Wikipedia by country {{as of|2022|12|lc=y}}. In grayed-out countries, a local language edition is usually the most popular.{{and then what|date=December 2025}}]]\n[[Wikipedia]] is a [[free content|free]] multilingual [[wiki]]-based [[open source|open-source]] online [[encyclopedia]] edited and maintained by [[Wikipedia community|a community of volunteer editors]], started on 15 January 2001 as [[English Wikipedia|an English-language encyclopedia]]. Non-English editions followed in the same year: the [[German Wikipedia|German]] and [[Catalan Wikipedia|Catalan edition]]s were created on 16 March,<ref>{{cite mailing list|title=[Wikipedia-l] Alternative language wikipedias|url=https://lists.wikimedia.org/pipermail/wikipedia-l/2001-March/000048.html|date=16 March 2001|ma

In [66]:
split = text.split(":|")
wikis = [t[-2:] for t in split[:-1]]
wikis = list(set(wikis))

print(len(wikis))
print(wikis)

279
['dm', 'zu', 'gd', 'uz', 'su', 'ny', 'ks', 'yi', 'nh', 'os', 'dr', 'hz', 'rr', 'so', 'ds', 'lv', 'kv', 'cl', 'ia', 'ln', 'sd', 'hn', 'my', 'rw', 'vk', 'sa', 'fa', 'oi', 'kj', 'pi', 'vi', 'lt', 'ls', 'cy', 'tt', 'bo', 'kb', 'qo', 'cg', 'da', 'nv', 'de', 'dc', 'uu', 'jv', 'um', 'ab', 'dz', 'ro', 'tn', 'rj', 'at', 'el', 'br', 'sv', 'be', 'ih', 'th', 'ak', 'km', 'bg', 'aj', 'bm', 'uw', 'np', 'eb', 'fo', 'ii', 'af', 'fy', 'bd', 'cu', 'zn', 'ly', 'wo', 'si', 'uk', 'wl', 'rv', 'ne', 'sh', 'ss', 'tg', 'nc', 'pa', 'nw', 'tl', 'sn', 'lk', 'ur', 'kg', 'zy', 'ry', 'tw', 'cn', 'kl', 'kw', 'ca', 'mh', 'ch', 'ot', 'sw', 'sk', 'mt', 'la', 'mi', 'gl', 'to', 'ff', 'na', 'nr', 'es', 'or', 'rm', 'ms', 'bh', 'ho', 'ay', 'sq', 'ig', 'ik', 'gn', 'bn', 'cd', 'om', 'ty', 'al', 'ce', 'wa', 'py', 'fi', 'ts', 'bi', 'as', 'dy', 'gh', 'sg', 'nn', 'io', 'lb', 'le', 'xr', 'zb', 'st', 'xh', 'ni', 'sl', 'fj', 'no', 'ug', 'ha', 'ht', 'ag', 'eu', 'tr', 'ue', 'uc', 'do', 'aw', 'rz', 'pe', 'ie', 'ap', 'dv', 'df', 'ka',

<h3>Next, get pageviews for each wiki in each language wiki domain</h3>

In [60]:
p = PageviewsClient(user_agent="cs315 student (nh107)")

In [ ]:
%%capture
#ignore error printing because there's too much

data = {}
for wiki in wikis:
    wiki_row = {}
    for title in wikipedia_titles:
        try:
            data = p.article_views(f'{wiki}.wikipedia', title, start='20260901', end='20260930')
            temp_df = pd.DataFrame(data)
            total_views = temp_df.sum(axis=1)
            wiki_row[title] = total_views
        except Exception:
            wiki_row[title] = None
    data[wiki] = wiki_row

df = pd.DataFrame.from_dict(data, orient='index')

Four_Hands,_Two_Sonatas    191537
dtype: int64
ERROR while fetching and parsing ['https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/en.wikipedia/all-access/all-agents/A_Shop_for_Killers/daily/2026090100/2026093000']


Traceback (most recent call last):
  File "C:\Users\chick\AppData\Roaming\Python\Python314\site-packages\mwviews\api\pageviews.py", line 136, in article_views
    results = self.get_concurrent(urls)
  File "C:\Users\chick\AppData\Roaming\Python\Python314\site-packages\mwviews\api\pageviews.py", line 423, in get_concurrent
    return list(executor.map(f, urls))
  File "c:\python314\Lib\concurrent\futures\_base.py", line 639, in result_iterator
    yield _result_or_cancel(fs.pop())
          ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "c:\python314\Lib\concurrent\futures\_base.py", line 311, in _result_or_cancel
    return fut.result(timeout)
           ~~~~~~~~~~^^^^^^^^^
  File "c:\python314\Lib\concurrent\futures\_base.py", line 445, in result
    self._condition.wait(timeout)
    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "c:\python314\Lib\threading.py", line 369, in wait
    waiter.acquire()
    ~~~~~~~~~~~~~~^^
KeyboardInterrupt


KeyboardInterrupt: 

In [68]:
df

,"Four Hands, Two Sonatas",A Shop for Killers,Agent Kim Reactivated,The Prosecutor's Proposal,Teach You a Lesson,The Wonderfools,The Legend of Kitchen Soldier,We Are All Trying Here,Yumi's Cells,Bloodhounds (TV series)
dm,None,None,None,None,None,None,None,None,None,None
zu,None,None,None,None,None,None,None,None,None,None
gd,None,None,None,None,None,None,None,None,None,None
uz,None,None,None,None,None,None,None,None,None,None
su,None,None,None,None,None,None,None,None,None,None


<h2>Does the popularity in Wikipedia pageviews correlate to the rating for the shows in MyDramaList?</h2>